# Privacy Filter — Korean — Test Notebook

Test the LoRA-fine-tuned `FrameByFrame/privacy-filter-korean` model on Korean and English PII detection.

**Capabilities (9 categories):**
- `private_person` — personal names (Korean / Western / handles)
- `private_address` — physical / postal addresses
- `private_phone` — phone numbers
- `private_email` — email addresses
- `private_date` — birthdays / personally-identifying dates
- `private_url` — personal URLs
- `account_number` — bank, card, RRN, passport, etc.
- `personal_handle` — usernames / handles
- `ip_address` — IP addresses

## 1. Install & Load Model

In [ ]:
# Install transformers from source. Required because Privacy Filter's
# `openai_privacy_filter` model_type is in transformers 5.x (HEAD) only.
#
# After this cell completes, RESTART THE RUNTIME (Runtime → Restart runtime
# on Colab, or Kernel → Restart on Jupyter), then run cell 3 onwards.

!pip install -q --upgrade "git+https://github.com/huggingface/transformers.git" peft safetensors accelerate

print()
print("=" * 60)
print("INSTALL DONE.")
print("Now: Runtime → Restart runtime, then run cell 3 onwards.")
print("=" * 60)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import time

MODEL_ID = "FrameByFrame/privacy-filter-korean"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16
)
model.eval()
if torch.cuda.is_available():
    model.cuda()

print(f"Model loaded. Categories: {sorted(set(v.split('-', 1)[-1] for v in model.config.id2label.values() if v != 'O'))}")

## 2. Helper — extract spans + show

In [ ]:
import json


def extract_pii(text: str, max_length: int = 512):
    """Run the model on `text` and decode BIOES into character-offset spans."""
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = logits.argmax(-1)[0].tolist()
    id2label = model.config.id2label

    spans = []
    active = None  # (label, start, end)
    for tok_idx, lid in enumerate(pred_ids):
        label = id2label[int(lid)]
        if label == "O":
            if active is not None:
                spans.append(active)
                active = None
            continue
        prefix, cat = label.split("-", 1)
        c_start, c_end = offsets[tok_idx]
        if prefix == "S":
            if active is not None:
                spans.append(active)
                active = None
            spans.append((cat, c_start, c_end))
        elif prefix == "B":
            if active is not None:
                spans.append(active)
            active = (cat, c_start, c_end)
        elif prefix in ("I", "E"):
            if active and active[0] == cat:
                active = (active[0], active[1], c_end)
            else:
                if active is not None:
                    spans.append(active)
                    active = None
                if prefix == "E":
                    spans.append((cat, c_start, c_end))
    if active is not None:
        spans.append(active)

    return [
        {"label": cat, "start": s, "end": e, "text": text[s:e].strip()}
        for cat, s, e in spans
        if text[s:e].strip()
    ]


def show(text: str):
    """Detect spans and pretty-print with timing."""
    t0 = time.time()
    spans = extract_pii(text)
    ms = round((time.time() - t0) * 1000)
    icon = "🚫" if spans else "✅"
    print(f"{icon} [{ms}ms] {text[:100]}")
    if spans:
        print(json.dumps(spans, indent=2, ensure_ascii=False))
    else:
        print("  (no PII detected)")
    print()


def redact(text: str) -> str:
    """Replace each detected span with [LABEL] in reverse order so offsets stay valid."""
    spans = sorted(extract_pii(text), key=lambda s: s["start"], reverse=True)
    out = text
    for s in spans:
        out = out[: s["start"]] + f"[{s['label'].upper()}]" + out[s["end"]:]
    return out

## 3. Korean — Chat-style PII

In [ ]:
show("김민수의 전화번호는 010-1234-5678이고 이메일은 minsu@example.com입니다.")
show("서울특별시 강남구 테헤란로 123에 사는 박지영씨에게 연락주세요.")
show("오늘 날씨가 좋네요.")  # safe — no PII
show("제 생일은 1990년 5월 14일입니다. 카드번호 1234-5678-9012-3456 잊지 마세요.")

## 4. Korean — Form-style document

This is the format-matched style (`이름:`, `주소:` clues) — Privacy Filter handles it strongly because the base model was trained on similar structured PII data.

In [ ]:
show("""고객 정보
이름: 이수진
생년월일: 1985년 3월 12일
주소: 부산광역시 해운대구 우동 1457
연락처: 010-9876-5432
이메일: lee.sj@daum.net""")

## 5. Korean — Banking / multi-PII

In [ ]:
show("신한은행 계좌번호 110-234-567890 (예금주 박민수), 등록 주소 인천광역시 연수구 송도과학로 100, 비상연락 010-2345-6789, 가입일 2018.04.22.")

## 6. English — names, addresses, accounts

In [ ]:
show("John Smith works at Google. Email: john@google.com, phone: 555-1234.")
show("Wire to acct 110-234-567890, contact minsu@example.com")
show("My SSN is 123-45-6789 and I live at 456 Oak Street, Springfield.")
show("The weather is nice today.")  # safe

## 7. Redaction

In [ ]:
samples = [
    "김민수님의 번호는 010-1234-5678입니다.",
    "서울특별시 강남구 테헤란로 123에 사는 박지영씨에게 연락주세요.",
    "My account is 110-234-567890 and email is minsu@example.com.",
]
for s in samples:
    print(f"  in: {s}")
    print(f" out: {redact(s)}")
    print()

## 8. Latency benchmark

In [ ]:
test_cases = [
    ("오늘 점심 뭐 먹지?", 0),
    ("010-1234-5678로 전화해", 1),
    ("What time is it?", 0),
    ("주민등록번호 901201-1234567", 1),
    ("김민수의 전화번호는 010-1234-5678이고 이메일은 minsu@example.com입니다.", 3),
    ("서울특별시 강남구 테헤란로 123에 사는 박지영씨에게 연락주세요.", 4),
    ("신한은행 계좌번호 110-234-567890 (예금주 박민수)", 2),
    ("My account is 110-234-567890 and email is minsu@example.com.", 2),
]

total_ms = 0
correct_count = 0
for text, expected_n in test_cases:
    t0 = time.time()
    spans = extract_pii(text)
    ms = round((time.time() - t0) * 1000)
    total_ms += ms
    n = len(spans)
    icon = "✅" if n == expected_n else ("~" if abs(n - expected_n) <= 1 else "❌")
    correct_count += int(n == expected_n)
    print(f"{icon} [{ms:>4}ms] expected={expected_n} got={n} | {text[:70]}")

print(f"\nExact-count match: {correct_count}/{len(test_cases)}")
print(f"Avg latency: {total_ms/len(test_cases):.0f}ms")

## 9. Custom Test

Try your own inputs:

In [ ]:
show("여기에 한국어 텍스트를 넣으세요")